# GEMS · GEOREGIME V1
### Multiscale GeoStructureNet · regional gravity context · two soft geological specialists

**One production pipeline, seven resumable fits, one NEW submission.** Uses the completed Geological Data Atlas from November 7, 2026. Three encoders (magnetic, gravity, terrain+structural evidence) feed a four-level gated multiscale backbone, a gravity-context bottleneck, one global score head and two lightly specialized score heads. A **label-independent, soft magnetic-vs-gravity router** learns where to weight specialists.

**Geological rationale.** The Atlas shows magnetics have relatively more energy at 0.2–10 km wavelengths (30.1%) than gravity (2.3%), while gravity is dominated by >30 km wavelengths (78.8%). Exploratory mapped-fault cluster experts reached only ~0.58 internal AUC: they are **not** used or imported here.

**Actual output:** 3 geographic CV runs + 3 withheld-fault-system runs + one full-label model, fold/split DTI proxies, cached checkpoints, output score/route GeoTIFFs, lineament proposal review, and a freshly trained continuous-confidence DrivenData TIFF.

This notebook has been authored and source-checked; **the actual Colab GPU runtime and DTI are not yet verified**. The high-resolution DEM Atlas stage did not finish, so no DEM channels are fabricated.


## 00 — One configuration

Defaults prioritize a real, resumable full experiment rather than an ablation sweep. Changing training values creates a new run directory; rerunning unchanged values resumes completed epochs.


In [ ]:
CONFIG = {
    "drive_root": "/content/drive/MyDrive/GEMS_2",
    "atlas_root": "GEMS_GEOLOGICAL_DATA_ATLAS_V1",
    "output_root": "GEMS_GEOREGIME_V1",
    "scratch_root": "/content/gems_georegime_stage",
    "seed": 2026,
    "patch": 256,
    "width": 24,
    "epochs": 8,
    "steps_per_epoch": 90,
    "batch_size": 4,
    "lr": 0.00035,
    "pu_prior": 0.015,
    "geo_buffer_px": 15,
    "discovery_buffer_px": 8,
    "predict_stride": 192,
    "use_amp": True,
}
RUN_GEOGRAPHIC = True
RUN_DISCOVERY = True
RUN_FINAL_EXPORT = True

REPO = "mybenkhadda/GEMS---drivendata"
REF = "main"
CODE = "/content/GEMS_GEOREGIME"


## 01 — Drive mount, pinned GitHub source and Colab dependencies

Only the existing Atlas feature stacks, its masks, and the original official submission template are required. Raw USGS DEMs are neither copied nor downloaded.


In [ ]:
from pathlib import Path
import subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
root = Path(CONFIG["drive_root"])
if not root.is_dir():
    raise RuntimeError("Drive mount inaccessible: use Runtime → Restart session and remount.")

if not (Path(CODE) / ".git").exists():
    subprocess.run(["git","clone","--depth","1","--branch",REF,
                    f"https://github.com/{REPO}.git",CODE],check=True)
else:
    subprocess.run(["git","-C",CODE,"fetch","--depth","1","origin",REF],check=True)
    subprocess.run(["git","-C",CODE,"checkout","--detach","FETCH_HEAD"],check=True)

subprocess.run([sys.executable,"-m","pip","install","-q","-e",CODE+"[colab]"],check=True)
if CODE not in sys.path: sys.path.insert(0,CODE)
HEAD=subprocess.check_output(["git","-C",CODE,"rev-parse","HEAD"],text=True).strip()
print("GEMS commit:",HEAD)


## 02 — Real-file preflight

Checks 33 selected source channels against the actual model-ready schema, validates five Atlas family GeoTIFFs, mapped-fault and eligibility rasters, CRS, affine transform, grid shape, and official submission template. Missing artifacts stop before training; no fabricated arrays or old model checkpoints.


In [ ]:
from gems_bench.georegime import GeoRegimeExperiment
EXP = GeoRegimeExperiment(CONFIG)
CHECKS = EXP.preflight()
print("Channels:",CHECKS["feature_channels"])
print("Mapped pixels:",CHECKS["mapped_pixels"])
print("Eligible pixels:",CHECKS["eligible_pixels"])


## 03 — Cached feature bank + 50–100 km gravity context

Select a compact geological feature bank from the Atlas rasters; store it as a fingerprinted, float16 memmap. Derive masked-normalized regional gravity fields using 5, 15 and 30 km Gaussian sigma across the full analysis region. These channels encode context beyond the 25.6 km local tile. The bank and context persist on Google Drive. Each fold later estimates robust normalization from **its training region only**.


In [ ]:
PREP = EXP.prepare()
print(PREP)
print("Input groups:",{k:len(v) for k,v in EXP.bank.group_indices.items()})
print("Features:",EXP.bank.names)


## 04 — Three geographic holdouts

Contiguous easting-quantile regions are held out in turn, with a 1.5 km excluded train buffer. Every fold trains a new shared GeoRegime model with two learned specialists. Predictions and checkpoints persist per fold. This is an offline **mapped-fault proxy DTI**, not a leaderboard score.


In [ ]:
if RUN_GEOGRAPHIC:
    GEO = EXP.run_geographic()
    print("Pooled geographic proxy DTI:", GEO)
else:
    print("Geographic validation skipped. Final export requires existing completed metrics.")


## 05 — Three withheld-fault-system discovery splits

Hold out complete 8-connected mapped fault traces, exclude their 800 m neighborhoods from training, then score within their 3 km surroundings. Retained mapped-fault pixels are excluded **pixel-exactly** from scoring. The feature bank and router are label-free; withheld fault labels never enter their inputs.


In [ ]:
if RUN_DISCOVERY:
    DISCOVERY = EXP.run_discovery()
    print("Pooled withheld-system proxy DTI:", DISCOVERY)
else:
    print("Discovery validation skipped. Final export requires existing completed metrics.")


## 06 — Validation components and interpretation

Report TP, FP and FN in addition to the DTI. Pooled DTI is calculated from **summed components**, not by averaging fold percentages. Note that positive/unlabeled training is an imperfect proxy for the organizer's hidden newly discovered faults.


In [ ]:
import pandas as pd
metrics=EXP.run_root/"tables"/"validation_components.csv"
if metrics.is_file():
    display(pd.read_csv(metrics).round(5))
else:
    print("No validation components available yet.")


## 07 — Full-label model, regional inference and NEW continuous submission

Once geographic/discovery reports are present, train an independent **full-label** GeoRegime model, predict all valid 100 m cells, and write a new continuous float32 probability GeoTIFF with the official template's CRS, affine transform and exact NaN footprint.

Atlas lineaments are produced as a separate label-free review raster/table. **No W2, E1, E6 or old cluster-classifier weights are loaded.** No top-1% binary masking is applied.


In [ ]:
if RUN_FINAL_EXPORT:
    OUTPUT=EXP.export()
    print("Submission status:",OUTPUT["status"])
    print("New GeoTIFF:",OUTPUT["file"])
    print("SHA256:",OUTPUT["sha256"])
else:
    print("Export disabled: no old-submission fallback.")


## 08 — Maps, router weights and convergence

Saves loss curves and full-footprint score/router maps as PNG/SVG. A nontrivial router map does not itself prove geological improvement. Inspect withheld-system DTI and the full confidence raster.


In [ ]:
EXP.figures()
print("Figures:", EXP.run_root/"figures")


## 09 — Exact artifacts and manual upload

Drive output root: /content/drive/MyDrive/GEMS_2/GEMS_GEOREGIME_V1/runs/\<config_hash\>

- checkpoints: geo0, geo1, geo2, disc0, disc1, disc2, full (each latest.pt and fold-specific normalizer)
- predictions: fold and discovery score rasters; full_prediction.tif; full_router_magnetic.tif; atlas_candidate_support.tif
- tables: DTI components, training histories, candidate_model_review.csv
- submissions: submission_georegime_v1.tif and its source/validation manifest

GPU work is **not** complete until this notebook has actually run and the final manifest says READY_FOR_MANUAL_UPLOAD.


In [ ]:
if RUN_FINAL_EXPORT and OUTPUT.get("status")=="READY_FOR_MANUAL_UPLOAD":
    from google.colab import files
    path=Path(OUTPUT["file"])
    assert path.is_file() and path.name=="submission_georegime_v1.tif"
    print("Download newly trained GeoRegime submission:",path)
    files.download(str(path))
else:
    print("No completed new model submission is ready.")


## Scientific and validation limitations

The mapped fault network is incomplete. The 3 discovery splits are pseudo-withheld systems sampled from existing mapped faults, not actual undiscovered labels; their DTI is only a proxy. The two model specialists are jointly learned from input geology, not the previous label-dependent clusters. No DEM is incorporated until the Atlas produces verified high-resolution features. Raw magnetic anomalies are not subjected to physically invalid flip augmentation. A leaderboard score remains unknown until the competition accepts a new submission.
